In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

FEATURE_COLS = ["accel_x", "accel_y", "accel_z"]
TIMESTEPS = 50
RANDOM_SEED = 42
TRAIN_PER_CLASS = 70
VAL_PER_CLASS = 15
TEST_PER_CLASS = 15

LABELS = [
    "still",
    "shake",
    "tilt forward",
    "tilt backward",
    "tilt left",
    "tilt right",
]
LABEL_TO_ID = {name: i for i, name in enumerate(LABELS)}
ID_TO_LABEL = {i: name for name, i in LABEL_TO_ID.items()}

In [2]:
def load_motion_samples(csv_path: str = "data.csv"):
    """Load CSV into (n_samples, 50, 3) arrays; timesteps stay in timestamp order."""
    df = pd.read_csv(csv_path)

    samples = []
    labels = []

    for sample_id, group in df.groupby("sample_id", sort=False):
        group = group.sort_values("timestamp")
        if len(group) != TIMESTEPS:
            raise ValueError(f"{sample_id} has {len(group)} rows, expected {TIMESTEPS}")

        samples.append(group[FEATURE_COLS].to_numpy(dtype=np.float32))
        labels.append(group["label"].iloc[0])

    X = np.stack(samples)
    y = np.array([LABEL_TO_ID[label] for label in labels], dtype=np.int32)
    return X, y


def balanced_stratified_split(X, y, seed=RANDOM_SEED):
    """70/15/15 split per class; shuffle samples within class, keep timesteps ordered."""
    rng = np.random.default_rng(seed)

    train_idx, val_idx, test_idx = [], [], []

    for class_id in range(len(LABELS)):
        class_indices = np.where(y == class_id)[0]
        if len(class_indices) != TRAIN_PER_CLASS + VAL_PER_CLASS + TEST_PER_CLASS:
            raise ValueError(
                f"class {ID_TO_LABEL[class_id]} has {len(class_indices)} samples, "
                f"expected {TRAIN_PER_CLASS + VAL_PER_CLASS + TEST_PER_CLASS}"
            )

        rng.shuffle(class_indices)
        train_idx.extend(class_indices[:TRAIN_PER_CLASS])
        val_idx.extend(class_indices[TRAIN_PER_CLASS : TRAIN_PER_CLASS + VAL_PER_CLASS])
        test_idx.extend(class_indices[TRAIN_PER_CLASS + VAL_PER_CLASS :])

    def take(indices):
        indices = np.array(indices)
        return X[indices], y[indices]

    return take(train_idx), take(val_idx), take(test_idx)


def to_tensors(X, y):
    return tf.constant(X, dtype=tf.float32), tf.constant(y, dtype=tf.int32)


def split_counts(y, name):
    unique, counts = np.unique(y, return_counts=True)
    print(f"{name}: {len(y)} samples")
    for label_id, count in zip(unique, counts):
        print(f"  {ID_TO_LABEL[label_id]}: {count}")

In [3]:
X, y = load_motion_samples("data.csv")
print(f"X shape: {X.shape}")  # (600, 50, 3)
print(f"y shape: {y.shape}")  # (600,)

(X_train, y_train), (X_val, y_val), (X_test, y_test) = balanced_stratified_split(X, y)

X_train, y_train = to_tensors(X_train, y_train)
X_val, y_val = to_tensors(X_val, y_val)
X_test, y_test = to_tensors(X_test, y_test)

split_counts(y_train.numpy(), "train")
split_counts(y_val.numpy(), "val")
split_counts(y_test.numpy(), "test")

X shape: (600, 50, 3)
y shape: (600,)
train: 420 samples
  still: 70
  shake: 70
  tilt forward: 70
  tilt backward: 70
  tilt left: 70
  tilt right: 70
val: 90 samples
  still: 15
  shake: 15
  tilt forward: 15
  tilt backward: 15
  tilt left: 15
  tilt right: 15
test: 90 samples
  still: 15
  shake: 15
  tilt forward: 15
  tilt backward: 15
  tilt left: 15
  tilt right: 15
